In [1]:
import numpy as np
import pandas as pd

%pip install -q kaggle

import os, warnings, random, math, glob, time
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error
from sklearn.linear_model import LinearRegression, HuberRegressor
warnings.filterwarnings("ignore")

Note: you may need to restart the kernel to use updated packages.


In [2]:
SEARCH_ROOTS = ["/kaggle/input", "/content", "/content/drive", "/mnt", "."]
_candidates = []
for root in SEARCH_ROOTS:
    if os.path.isdir(root):
        _candidates += glob.glob(f"{root}/**/diabetes_subset_sensor_data", recursive=True)
assert _candidates, "Dataset not found — check it's attached."
DATA_DIR = os.path.dirname(sorted(_candidates, key=len)[0])
OUTPUT_DIR = "/kaggle/working" if os.path.isdir("/kaggle/working") else "/content/outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)
print("DATA_DIR:", DATA_DIR)

DATA_DIR: /kaggle/input/datasets/sarabhian/d1namo-ecg-glucose-data


In [3]:
WINDOW_SIZE   = 48 #past 4 hours
HORIZONS      = [3, 6, 12]   # t+15, t+30, t+60 (5-min sampling)
SENSOR_COLS   = ["HR", "Activity", "BR", "PeakAccel",
                 "recent_carbs", "recent_units", "cgm_roc", "cgm_accel"]  # SkinTemp excluded — dead sensor
SKIN_TEMP_MIN, SKIN_TEMP_MAX = -40.0, 60.0
TOLERANCE_SEC = 90

#train-70%
#validation-10%
#testing-remaining 20%
TRAIN_FRAC, VAL_FRAC = 0.70, 0.10

#file handling
def resolve(*parts):
    p = os.path.join(DATA_DIR, *parts)
    inner = os.path.join(p, parts[0])
    return inner if os.path.isdir(inner) else p


#access sensor and glucose sets
subsets = {"diabetes": {"sensor": resolve("diabetes_subset_sensor_data"),
                        "glucose": resolve("diabetes_subset_pictures-glucose-food-insulin")}}

#this list will hold all the subjects found in the dataset (glucose and sensor data)
all_subjects = []
for group, paths in subsets.items():
    for subj_id in sorted(os.listdir(paths["sensor"])):
        sensor_path  = os.path.join(paths["sensor"], subj_id)
        glucose_path = os.path.join(paths["glucose"], subj_id, "glucose.csv")
        if os.path.isdir(sensor_path) and os.path.exists(glucose_path):
            all_subjects.append({"id": f"{group}_{subj_id}", "group": group,
                                  "sensor_path": sensor_path, "glucose_csv": glucose_path})

            print(f"Subjects found: {len(all_subjects)}")

Subjects found: 1
Subjects found: 2
Subjects found: 3
Subjects found: 4
Subjects found: 5
Subjects found: 6
Subjects found: 7
Subjects found: 8
Subjects found: 9


In [23]:
for s in all_subjects[:1]:
    folder = os.path.dirname(s["glucose_csv"])
    print(os.listdir(folder))

['food.csv', 'food_pictures', 'glucose.csv', 'insulin.csv']


In [24]:
food_data = {}
insulin_data = {}

for s in all_subjects:
    folder = os.path.dirname(s["glucose_csv"])

    food_path = os.path.join(folder, "food.csv")
    insulin_path = os.path.join(folder, "insulin.csv")

    if os.path.exists(food_path):
        df = pd.read_csv(food_path)
        df["patient"] = s["id"]
        food_data[s["id"]] = df

    if os.path.exists(insulin_path):
        df = pd.read_csv(insulin_path)
        df["patient"] = s["id"]
        insulin_data[s["id"]] = df

food_df = pd.concat(food_data.values(), ignore_index=True)
insulin_df = pd.concat(insulin_data.values(), ignore_index=True)

print("Food records:", len(food_df))
print("Insulin records:", len(insulin_df))

display(food_df.head())
display(insulin_df.head())

Food records: 115
Insulin records: 126


,picture,description,calories,balance,quality,datetime,patient
0,001.jpg,Iceberg lettuce with olive oil and vinegar,357.0,Unbalance,Good quality,2014:10:01 19:27:49,diabetes_001
1,002.jpg,"Tuna salade with boiled egg, asiago cheese an...",866.0,Unbalance,Good quality,2014:10:01 19:32:00,diabetes_001
2,003.jpg,Asiago cheese,190.0,Unbalance,Good quality,2014:10:01 19:43:59,diabetes_001
3,004.jpg,"Lemon yogurt, asiago cheese and bread with kiwi",1063.0,Balance,Good quality,2014:10:02 10:11:16,diabetes_001
4,005.jpg,Tomatoes with cheese and bread,979.0,Unbalance,Good quality,2014:10:02 19:34:03,diabetes_001


,date,time,fast_insulin,slow_insulin,comment,patient
0,2014-10-01,10:06:00,7.0,NaN,NaN,diabetes_001
1,2014-10-01,16:50:00,4.0,NaN,NaN,diabetes_001
2,2014-10-01,19:28:00,6.0,NaN,NaN,diabetes_001
3,2014-10-01,22:27:00,8.0,NaN,NaN,diabetes_001
4,2014-10-01,23:48:00,0.0,31.0,NaN,diabetes_001


In [4]:
#preprocessing glucose and sensor (summary csv files)
def load_glucose(glucose_csv):
    df = pd.read_csv(glucose_csv)
    df = df[df["type"] == "cgm"].copy()
    df["datetime"] = pd.to_datetime(df["date"] + " " + df["time"])
    return df.sort_values("datetime").reset_index(drop=True)[["datetime", "glucose"]]

def load_sensor(sensor_path):
    sensor_dir = os.path.join(sensor_path, "sensor_data")
    if not os.path.isdir(sensor_dir):
        return None
    all_dfs = []
    for session in sorted(os.listdir(sensor_dir)):
        fpath = os.path.join(sensor_dir, session, f"{session}_Summary.csv")
        if os.path.exists(fpath):
            all_dfs.append(pd.read_csv(fpath))
    if not all_dfs:
        return None
    combined = pd.concat(all_dfs, ignore_index=True)
    combined["datetime"] = pd.to_datetime(combined["Time"], format="%d/%m/%Y %H:%M:%S.%f", errors="coerce")
    combined = combined.dropna(subset=["datetime"]).sort_values("datetime")
    base_cols = ["HR", "Activity", "PeakAccel", "BR"] ############### used columns ###########
    cols = ["datetime"] + [c for c in base_cols if c in combined.columns]
    return combined[cols].reset_index(drop=True)

In [5]:
#only glucose
subject_merged = {}
for s in all_subjects:
    gdf = load_glucose(s["glucose_csv"])
    if len(gdf) < WINDOW_SIZE + max(HORIZONS):
        continue
    subject_merged[s["id"]] = gdf
    gdf["patient"] = s["id"]  # Add patient identifier

glucose_df = pd.concat(subject_merged.values(), ignore_index=True)
glucose_df #all patients all time

,datetime,glucose,patient
0,2014-10-01 19:14:00,10.3,diabetes_001
1,2014-10-01 19:19:00,9.9,diabetes_001
2,2014-10-01 19:24:00,9.8,diabetes_001
3,2014-10-01 19:29:00,9.6,diabetes_001
4,2014-10-01 19:34:00,9.4,diabetes_001
...,...,...,...
8050,2014-10-03 12:20:19,4.9,diabetes_009
8051,2014-10-03 12:25:19,4.2,diabetes_009
8052,2014-10-03 12:30:19,3.3,diabetes_009
8053,2014-10-03 12:35:19,2.7,diabetes_009


In [6]:
#find time frequency of glucose readings
glucose_df = glucose_df.sort_values(["patient", "datetime"])

glucose_df["time_diff"] = (
    glucose_df.groupby("patient")["datetime"]
    .diff()
    .dt.total_seconds() / 60 #convert to minutes
)

#glucose_df['time_diff'].value_counts()

In [7]:
glucose_df.groupby("patient")["time_diff"].describe()

,count,mean,std,min,25%,50%,75%,max
patient,,,,,,,,
diabetes_001,1412.0,5.000024,0.000627,5.0,5.0,5.0,5.0,5.016667
diabetes_002,1055.0,5.000032,0.000725,5.0,5.0,5.0,5.0,5.016667
diabetes_003,182.0,5.000092,0.001235,5.0,5.0,5.0,5.0,5.016667
diabetes_004,968.0,5.000034,0.000757,5.0,5.0,5.0,5.0,5.016667
diabetes_005,908.0,5.000037,0.000782,5.0,5.0,5.0,5.0,5.016667
diabetes_006,1279.0,5.000026,0.000659,5.0,5.0,5.0,5.0,5.016667
diabetes_007,987.0,5.339446,10.663166,5.0,5.0,5.0,5.0,340.000000
diabetes_008,1139.0,5.000029,0.000698,5.0,5.0,5.0,5.0,5.016667
diabetes_009,116.0,15.775862,116.059586,5.0,5.0,5.0,5.0,1255.000000


In [8]:
#glucose_df.loc[glucose_df['glucose'] < 3.9]  #hypoglycemia threshold
#glucose_df.loc[glucose_df['glucose'] > 11.1]  #hyperglycemia threshold

In [9]:
#only sensor readings
subject_merged = {}
for s in all_subjects:
    sdf = load_sensor(s["sensor_path"])
    if len(sdf) < WINDOW_SIZE + max(HORIZONS):
        continue
    subject_merged[s["id"]] = sdf
    sdf["patient"] = s["id"]  # Add patient identifier

sensor_df = pd.concat(subject_merged.values(), ignore_index=True)
sensor_df #all patients all time

,datetime,HR,Activity,PeakAccel,BR,patient
0,2014-10-01 10:09:39.417,65,0.41,0.84,8.1,diabetes_001
1,2014-10-01 10:09:40.417,65,0.51,1.03,8.1,diabetes_001
2,2014-10-01 10:09:41.417,65,0.30,0.74,7.3,diabetes_001
3,2014-10-01 10:09:42.417,65,0.43,0.98,7.3,diabetes_001
4,2014-10-01 10:09:43.417,65,0.49,1.10,6.6,diabetes_001
...,...,...,...,...,...,...
1622568,2014-10-04 22:45:39.428,0,0.37,0.63,6553.5,diabetes_009
1622569,2014-10-04 22:45:40.428,0,0.42,0.72,9.8,diabetes_009
1622570,2014-10-04 22:45:41.428,0,0.30,0.34,6553.5,diabetes_009
1622571,2014-10-04 22:45:42.428,0,0.18,0.26,6553.5,diabetes_009


In [10]:
#find time frequency of glucose readings
sensor_df = sensor_df.sort_values("datetime")

sensor_df = sensor_df.sort_values(["patient", "datetime"]).copy()

sensor_df["time_diff"] = (
    sensor_df.groupby("patient")["datetime"]
    .diff()
    .dt.total_seconds() / 60
)

In [11]:
sensor_df.groupby("patient")["time_diff"].describe()

,count,mean,std,min,25%,50%,75%,max
patient,,,,,,,,
diabetes_001,154485.0,0.029377,2.911570,0.016667,0.016667,0.016667,0.016667,753.316717
diabetes_002,167824.0,0.028443,2.656824,0.016667,0.016667,0.016667,0.016667,635.116683
diabetes_003,175961.0,0.027714,2.726407,0.016667,0.016667,0.016667,0.016667,790.083367
diabetes_004,157199.0,0.031621,3.359890,0.016667,0.016667,0.016667,0.016667,865.666000
diabetes_005,168620.0,0.027367,2.527668,0.016667,0.016667,0.016667,0.016667,656.533350
diabetes_006,201254.0,0.025989,2.333946,0.016667,0.016667,0.016667,0.016667,695.783350
diabetes_007,214294.0,0.023922,2.023879,0.016667,0.016667,0.016667,0.016667,721.850050
diabetes_008,193636.0,0.022734,1.942492,0.016667,0.016667,0.016667,0.016667,729.733367
diabetes_009,189291.0,0.028138,2.885792,0.016650,0.016667,0.016667,0.016667,779.700017


In [12]:
# Inspect HR readings around zero values for all patients

zero_hr = sensor_df[
    sensor_df["HR"] == 0
].sort_values(["patient", "datetime"])

zero_hr[
    ["patient", "datetime", "HR", "BR", "Activity"]
]

,patient,datetime,HR,BR,Activity
7,diabetes_001,2014-10-01 10:09:46.417,0,7.0,0.07
8,diabetes_001,2014-10-01 10:09:47.417,0,7.3,0.07
9,diabetes_001,2014-10-01 10:09:48.417,0,7.3,0.06
10,diabetes_001,2014-10-01 10:09:49.417,0,7.6,0.05
11,diabetes_001,2014-10-01 10:09:50.417,0,7.6,0.03
...,...,...,...,...,...
1622568,diabetes_009,2014-10-04 22:45:39.428,0,6553.5,0.37
1622569,diabetes_009,2014-10-04 22:45:40.428,0,9.8,0.42
1622570,diabetes_009,2014-10-04 22:45:41.428,0,6553.5,0.30
1622571,diabetes_009,2014-10-04 22:45:42.428,0,6553.5,0.18


In [13]:
hr_zero_summary = sensor_df.groupby("patient").agg(
    total_readings=("HR", "count"),
    zero_hr_count=("HR", lambda x: (x == 0).sum())
)

hr_zero_summary["zero_hr_percentage"] = (
    hr_zero_summary["zero_hr_count"] /
    hr_zero_summary["total_readings"] * 100
)

hr_zero_summary.sort_values(
    "zero_hr_percentage",
    ascending=False
)

,total_readings,zero_hr_count,zero_hr_percentage
patient,,,
diabetes_005,168621,105932,62.822543
diabetes_009,189292,118654,62.683051
diabetes_003,175962,74425,42.296064
diabetes_001,154486,63572,41.150654
diabetes_004,157200,62168,39.547074
diabetes_006,201255,58595,29.114805
diabetes_002,167825,28951,17.250708
diabetes_008,193637,11649,6.015896
diabetes_007,214295,132,0.061597


In [14]:
# Identify consecutive periods where HR = 0

sensor_df = sensor_df.sort_values(
    ["patient", "datetime"]
).copy()

sensor_df["hr_zero"] = sensor_df["HR"] == 0

# Identify when the zero/nonzero status changes
sensor_df["hr_group"] = (
    sensor_df.groupby("patient")["hr_zero"]
    .transform(lambda x: x.ne(x.shift()).cumsum())
)

# Summarize consecutive zero periods
zero_periods = (
    sensor_df[sensor_df["hr_zero"]]
    .groupby(["patient", "hr_group"])
    .agg(
        start_time=("datetime", "min"),
        end_time=("datetime", "max"),
        readings=("HR", "size")
    )
    .reset_index()
)

zero_periods["duration_seconds"] = (
    zero_periods["end_time"] - zero_periods["start_time"]
).dt.total_seconds()

zero_periods.sort_values(
    "readings",
    ascending=False
)

,patient,hr_group,start_time,end_time,readings,duration_seconds
2474,diabetes_003,710,2014-10-02 16:28:53.430,2014-10-03 04:56:10.430,44838,44837.0
10936,diabetes_009,6800,2014-10-04 19:29:33.428,2014-10-04 22:45:17.428,11745,11744.0
7535,diabetes_008,36,2014-10-04 04:55:08.428,2014-10-04 08:00:33.428,11126,11125.0
2358,diabetes_003,478,2014-10-01 15:04:28.428,2014-10-01 17:04:48.428,7221,7220.0
10934,diabetes_009,6796,2014-10-04 17:30:35.428,2014-10-04 19:26:41.428,6967,6966.0
...,...,...,...,...,...,...
10872,diabetes_009,6672,2014-10-04 16:25:27.428,2014-10-04 16:25:27.428,1,0.0
5608,diabetes_005,3058,2014-10-03 11:50:15.429,2014-10-03 11:50:15.429,1,0.0
5619,diabetes_005,3080,2014-10-03 12:07:39.429,2014-10-03 12:07:39.429,1,0.0
5620,diabetes_005,3082,2014-10-03 12:08:30.429,2014-10-03 12:08:30.429,1,0.0


In [15]:
# Check overlapping unusual readings

hr_zero = sensor_df["HR"] == 0
br_unusual = sensor_df["BR"] == 6553.5

print("Total HR zero readings:", hr_zero.sum())
print("Total unusual BR readings:", br_unusual.sum())

print("Both occurring together:", (hr_zero & br_unusual).sum())

print(
    "Percentage of HR zero readings with unusual BR:",
    (hr_zero & br_unusual).sum() / hr_zero.sum() * 100
)

Total HR zero readings: 524078
Total unusual BR readings: 224
Both occurring together: 140
Percentage of HR zero readings with unusual BR: 0.02671358080285759


In [16]:
# Find timestamps where both occur

both_unusual = sensor_df[
    (sensor_df["HR"] == 0) &
    (sensor_df["BR"] == 6553.5)
]

both_unusual[
    ["patient", "datetime", "HR", "BR", "Activity"]
]

,patient,datetime,HR,BR,Activity
128441,diabetes_001,2014-10-03 19:18:58.421,0,6553.5,1.16
128442,diabetes_001,2014-10-03 19:18:59.421,0,6553.5,0.68
128443,diabetes_001,2014-10-03 19:19:00.421,0,6553.5,0.39
128444,diabetes_001,2014-10-03 19:19:01.421,0,6553.5,0.23
128445,diabetes_001,2014-10-03 19:19:02.421,0,6553.5,0.17
...,...,...,...,...,...
1622567,diabetes_009,2014-10-04 22:45:38.428,0,6553.5,0.65
1622568,diabetes_009,2014-10-04 22:45:39.428,0,6553.5,0.37
1622570,diabetes_009,2014-10-04 22:45:41.428,0,6553.5,0.30
1622571,diabetes_009,2014-10-04 22:45:42.428,0,6553.5,0.18


### Combined Glucose and Sensor data by time

In [17]:
#align sensor and glucose data based on timestamps
def align_glucose_sensor(glucose_df, sensor_df):
    merged = pd.merge_asof(
        glucose_df.sort_values("datetime"), sensor_df.sort_values("datetime"),
        on="datetime", tolerance=pd.Timedelta(seconds=TOLERANCE_SEC), direction="nearest")
    return merged.dropna(subset=["glucose"]).reset_index(drop=True)

In [18]:
subject_merged = {}
for s in all_subjects:
    gdf = load_glucose(s["glucose_csv"]); sdf = load_sensor(s["sensor_path"])
    sdf["patient"] = s["id"]  # Add patient identifier
    if sdf is None or len(gdf) < WINDOW_SIZE + max(HORIZONS):
        continue
    merged = align_glucose_sensor(gdf, sdf)   #for one subject at a time
    subject_merged[s["id"]] = merged

subj_ids = list(subject_merged.keys())
print("Subjects with usable data:", len(subj_ids))

Subjects with usable data: 9


In [19]:
merged_df = pd.concat(subject_merged.values(), ignore_index=True)
merged_df

,datetime,glucose,HR,Activity,PeakAccel,BR,patient
0,2014-10-01 19:14:00,10.3,68.0,0.01,0.04,21.5,diabetes_001
1,2014-10-01 19:19:00,9.9,87.0,0.01,0.06,15.9,diabetes_001
2,2014-10-01 19:24:00,9.8,75.0,0.06,0.10,13.3,diabetes_001
3,2014-10-01 19:29:00,9.6,93.0,0.13,0.22,8.6,diabetes_001
4,2014-10-01 19:34:00,9.4,95.0,0.24,0.54,18.8,diabetes_001
...,...,...,...,...,...,...,...
8050,2014-10-03 12:20:19,4.9,0.0,0.06,0.08,19.4,diabetes_009
8051,2014-10-03 12:25:19,4.2,0.0,0.03,0.06,20.1,diabetes_009
8052,2014-10-03 12:30:19,3.3,36.0,0.24,0.26,17.9,diabetes_009
8053,2014-10-03 12:35:19,2.7,34.0,0.08,0.19,19.9,diabetes_009


In [20]:
# Sort by patient and timestamp
merged_df = merged_df.sort_values(
    ["patient", "datetime"]
).copy()

# Calculate time difference in minutes for each patient
merged_df["time_diff"] = (
    merged_df.groupby("patient")["datetime"]
    .diff()
    .dt.total_seconds() / 60
)

# Summary statistics by patient
sampling_summary = merged_df.groupby("patient")["time_diff"].describe()

sampling_summary

,count,mean,std,min,25%,50%,75%,max
patient,,,,,,,,
diabetes_001,408.0,9.791708,56.260952,5.0,5.0,5.0,5.0,755.000000
diabetes_002,482.0,9.056086,49.390591,5.0,5.0,5.0,5.0,635.000000
diabetes_003,160.0,5.000104,0.001318,5.0,5.0,5.0,5.0,5.016667
diabetes_004,501.0,9.660745,59.052801,5.0,5.0,5.0,5.0,870.000000
diabetes_005,550.0,8.254606,44.098419,5.0,5.0,5.0,5.0,660.000000
diabetes_006,492.0,8.790718,46.790150,5.0,5.0,5.0,5.0,695.000000
diabetes_007,611.0,8.085161,40.063134,5.0,5.0,5.0,5.0,725.016667
diabetes_008,646.0,6.811197,33.512592,5.0,5.0,5.0,5.0,735.000000
diabetes_009,114.0,16.008772,117.541519,5.0,5.0,5.0,5.0,1260.000000


In [21]:
merged_df = merged_df.sort_values("datetime")

merged_df['time_diff'] = merged_df["datetime"].diff()
merged_df['time_diff'] = merged_df['time_diff'].dt.total_seconds() / 60 #difference in minutes
 

In [22]:
merged_df.groupby("patient")["time_diff"].describe()

,count,mean,std,min,25%,50%,75%,max
patient,,,,,,,,
diabetes_001,409.0,1.864466,0.364651,0.733333,1.983333,1.983333,1.983333,1.983333
diabetes_002,483.0,0.548516,0.765109,0.100000,0.100000,0.100000,1.800000,1.933333
diabetes_003,161.0,1.326190,0.045480,1.233333,1.350000,1.350000,1.350000,1.350000
diabetes_004,502.0,1.136687,0.377888,0.933333,0.933333,0.933333,0.950000,3.033333
diabetes_005,551.0,0.123261,0.052299,0.083333,0.083333,0.100000,0.116667,0.266667
diabetes_006,493.0,0.124983,0.019263,0.100000,0.116667,0.116667,0.133333,0.166667
diabetes_007,612.0,0.143301,0.281753,0.033333,0.050000,0.050000,0.050000,1.000000
diabetes_008,647.0,1.826662,0.585366,1.600000,1.700000,1.716667,1.716667,5.000000
diabetes_009,115.0,0.109130,0.008331,0.100000,0.100000,0.116667,0.116667,0.116667


could time parse to every 5 min, 10 min, ...